# Notebook 8: Machine Learning Modelling

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Modelling Period:** October 2017 – December 2025

## Notebook Objective

This notebook develops machine learning models for predicting monthly food inflation across South African food subclasses.

The analysis will:

1. load the modelling dataset prepared in Notebook 5
2. preserve the predefined chronological data splits
3. compare symmetric and asymmetric exchange-rate feature representations
4. establish persistence and regularised linear benchmarks
5. train Random Forest and XGBoost models
6. select model settings using the validation period only
7. prevent the test period from influencing model development
8. export fitted models and predictions for final evaluation in Notebook 9

The target variable is monthly food inflation. The symmetric models use lagged overall exchange-rate changes, while the asymmetric models use separate lagged depreciation and appreciation features.

Notebook 8 performs model development and validation. Final test evaluation and comparison with the econometric benchmarks are reserved for Notebook 9.

## Machine Learning Strategy

A pooled modelling approach is used across the 46 food subclasses. Each
observation represents one food subclass in one month.

Food-subclass indicators allow the models to account for persistent
category-level differences, while lagged food-inflation and exchange-rate features capture temporal relationships.

Three model families will be considered:

- Ridge regression as a regularised linear benchmark
- Random Forest for nonlinear relationships and interactions
- XGBoost for gradient-boosted tree modelling

A persistence forecast, which uses the previous month's food inflation as the prediction, provides a simple time-series benchmark.

Each model family will be estimated using two feature representations:

- a symmetric representation using lagged total exchange-rate changes
- an asymmetric representation using separate depreciation and appreciation magnitudes.

Models will be trained on observations ending in December 2023. Hyperparameter selection will use the 2024 validation period. The 2025 test period will remain untouched during model development.

In [68]:
# import Libraries

from pathlib import Path

import joblib
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

RANDOM_STATE = 42

print("Libraries imported successfully.")
print("scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgboost.__version__)

Libraries imported successfully.
scikit-learn version: 1.9.0
XGBoost version: 3.3.0


In [69]:
# load and validate the data handoff

ml_data_path = Path(
    "../data/processed/ml_model_data.csv"
)

if not ml_data_path.exists():
    raise FileNotFoundError(
        f"Machine learning dataset not found: {ml_data_path}"
    )

ml_data = pd.read_csv(
    ml_data_path,
    parse_dates=["Date"],
)

ml_data = (
    ml_data
    .sort_values(["Date", "SubclassDescription"])
    .reset_index(drop=True)
)

required_handoff_columns = {
    "Date",
    "ClassDescription",
    "SubclassDescription",
    "Subclass_Weight",
    "Food_Inflation_Pct",
    "Split",
}

missing_handoff_columns = (
    required_handoff_columns.difference(ml_data.columns)
)

if missing_handoff_columns:
    raise ValueError(
        "Missing handoff columns: "
        f"{sorted(missing_handoff_columns)}"
    )

handoff_summary = pd.DataFrame(
    {
        "Value": [
            len(ml_data),
            ml_data["SubclassDescription"].nunique(),
            ml_data["Date"].nunique(),
            ml_data["Date"].min(),
            ml_data["Date"].max(),
            ml_data.isna().sum().sum(),
            ml_data.duplicated(
                ["Date", "SubclassDescription"]
            ).sum(),
        ]
    },
    index=[
        "Observations",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Missing values",
        "Duplicate subclass-month rows",
    ],
)

display(handoff_summary)

,Value
Observations,4554
Food subclasses,46
Unique months,99
Start date,2017-10-01 00:00:00
End date,2025-12-01 00:00:00
Missing values,0
Duplicate subclass-month rows,0


In [70]:
# define target and features

target_column = "Food_Inflation_Pct"

categorical_features = [
    "SubclassDescription",
]

shared_numeric_features = [
    "Food_Inflation_Lag1_Pct",
    "Food_Inflation_Lag3_Pct",
    "Food_Inflation_Lag6_Pct",
    "Month_Sin",
    "Month_Cos",
]

symmetric_exchange_features = [
    "ExchangeRate_Change_Lag1_Pct",
    "ExchangeRate_Change_Lag3_Pct",
    "ExchangeRate_Change_Lag6_Pct",
]

asymmetric_exchange_features = [
    "Depreciation_Shock_Lag1_Pct",
    "Depreciation_Shock_Lag3_Pct",
    "Depreciation_Shock_Lag6_Pct",
    "Appreciation_Magnitude_Lag1_Pct",
    "Appreciation_Magnitude_Lag3_Pct",
    "Appreciation_Magnitude_Lag6_Pct",
]

symmetric_model_features = (
    categorical_features
    + shared_numeric_features
    + symmetric_exchange_features
)

asymmetric_model_features = (
    categorical_features
    + shared_numeric_features
    + asymmetric_exchange_features
)

required_model_columns = set(
    symmetric_model_features
    + asymmetric_model_features
    + [target_column]
)

missing_model_columns = required_model_columns.difference(
    ml_data.columns
)

if missing_model_columns:
    raise ValueError(
        "Missing modelling columns: "
        f"{sorted(missing_model_columns)}"
    )

feature_summary = pd.DataFrame(
    {
        "Representation": [
            "Symmetric",
            "Asymmetric",
        ],
        "Categorical_Features": [
            len(categorical_features),
            len(categorical_features),
        ],
        "Numeric_Features": [
            (
                len(shared_numeric_features)
                + len(symmetric_exchange_features)
            ),
            (
                len(shared_numeric_features)
                + len(asymmetric_exchange_features)
            ),
        ],
        "Total_Input_Columns": [
            len(symmetric_model_features),
            len(asymmetric_model_features),
        ],
    }
)

display(feature_summary)

,Representation,Categorical_Features,Numeric_Features,Total_Input_Columns
0,Symmetric,1,8,9
1,Asymmetric,1,11,12


In [71]:
# create chronological data splits

split_order = ["Train", "Validation", "Test"]

unexpected_splits = set(
    ml_data["Split"].unique()
).difference(split_order)

if unexpected_splits:
    raise ValueError(
        f"Unexpected split labels: {sorted(unexpected_splits)}"
    )

train_data = ml_data.loc[
    ml_data["Split"] == "Train"
].copy()

validation_data = ml_data.loc[
    ml_data["Split"] == "Validation"
].copy()

test_data = ml_data.loc[
    ml_data["Split"] == "Test"
].copy()

if not (
    train_data["Date"].max()
    < validation_data["Date"].min()
    <= validation_data["Date"].max()
    < test_data["Date"].min()
):
    raise ValueError(
        "The chronological split boundaries are invalid."
    )

split_summary = (
    ml_data
    .groupby("Split")
    .agg(
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Observations=("Date", "size"),
        Food_Subclasses=(
            "SubclassDescription",
            "nunique",
        ),
        Unique_Months=("Date", "nunique"),
    )
    .reindex(split_order)
)

display(split_summary)

print(
    "Training ends before validation:",
    train_data["Date"].max()
    < validation_data["Date"].min(),
)

print(
    "Validation ends before testing:",
    validation_data["Date"].max()
    < test_data["Date"].min(),
)

,Start_Date,End_Date,Observations,Food_Subclasses,Unique_Months
Split,,,,,
Train,2017-10-01,2023-12-01,3450,46,75
Validation,2024-01-01,2024-12-01,552,46,12
Test,2025-01-01,2025-12-01,552,46,12


Training ends before validation: True
Validation ends before testing: True


### Modelling Data Interpretation

The modelling dataset contains 99 monthly observations for each of the 46 food subclasses.

The training period provides 75 months per subclass, while the validation and test periods each contain 12 months per subclass. Every food subclass is represented in all three periods.

The symmetric representation contains eight numeric predictors and one
categorical predictor. The asymmetric representation contains three additional numeric predictors because depreciation and appreciation are represented separately.

The larger asymmetric feature set will be compared with the symmetric set within each model family. This ensures that any improvement is attributed to the exchange-rate representation rather than to a different algorithm or evaluation period.

In [72]:
# create training and validation matrices

model_feature_sets = {
    "Symmetric": symmetric_model_features,
    "Asymmetric": asymmetric_model_features,
}

numeric_feature_sets = {
    "Symmetric": (
        shared_numeric_features
        + symmetric_exchange_features
    ),
    "Asymmetric": (
        shared_numeric_features
        + asymmetric_exchange_features
    ),
}

training_features = {
    representation: train_data[features].copy()
    for representation, features in model_feature_sets.items()
}

validation_features = {
    representation: validation_data[features].copy()
    for representation, features in model_feature_sets.items()
}

training_target = train_data[target_column].copy()
validation_target = validation_data[target_column].copy()

training_subclasses = set(
    train_data["SubclassDescription"]
)
validation_subclasses = set(
    validation_data["SubclassDescription"]
)
test_subclasses = set(
    test_data["SubclassDescription"]
)

print(
    "Training target observations:",
    len(training_target),
)
print(
    "Validation target observations:",
    len(validation_target),
)
print(
    "Validation subclasses unseen during training:",
    len(validation_subclasses - training_subclasses),
)
print(
    "Test subclasses unseen during training:",
    len(test_subclasses - training_subclasses),
)

Training target observations: 3450
Validation target observations: 552
Validation subclasses unseen during training: 0
Test subclasses unseen during training: 0


In [73]:
# define preprocessing pipelines

def create_preprocessor(
    numeric_features,
    scale_numeric,
):
    numeric_transformer = (
        StandardScaler()
        if scale_numeric
        else "passthrough"
    )

    return ColumnTransformer(
        transformers=[
            (
                "category",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_features,
            ),
            (
                "numeric",
                numeric_transformer,
                numeric_features,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


ridge_preprocessors = {
    representation: create_preprocessor(
        numeric_features=numeric_features,
        scale_numeric=True,
    )
    for representation, numeric_features
    in numeric_feature_sets.items()
}

tree_preprocessors = {
    representation: create_preprocessor(
        numeric_features=numeric_features,
        scale_numeric=False,
    )
    for representation, numeric_features
    in numeric_feature_sets.items()
}

preprocessing_records = []

for representation in model_feature_sets:
    fitted_preprocessor = clone(
        ridge_preprocessors[representation]
    )

    transformed_training_data = (
        fitted_preprocessor.fit_transform(
            training_features[representation]
        )
    )

    preprocessing_records.append(
        {
            "Representation": representation,
            "Input_Columns": len(
                model_feature_sets[representation]
            ),
            "Transformed_Features": (
                transformed_training_data.shape[1]
            ),
            "Training_Observations": (
                transformed_training_data.shape[0]
            ),
        }
    )

preprocessing_summary = pd.DataFrame(
    preprocessing_records
)

display(preprocessing_summary)

,Representation,Input_Columns,Transformed_Features,Training_Observations
0,Symmetric,9,54,3450
1,Asymmetric,12,57,3450


In [74]:
# evaluate the validation benchmark

def calculate_regression_metrics(
    actual_values,
    predicted_values,
):
    actual = np.asarray(actual_values, dtype=float)
    predicted = np.asarray(
        predicted_values,
        dtype=float,
    )

    return {
        "MAE": mean_absolute_error(
            actual,
            predicted,
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual,
                predicted,
            )
        ),
        "R2": r2_score(
            actual,
            predicted,
        ),
        "Directional_Accuracy_Pct": (
            np.mean(
                np.sign(actual)
                == np.sign(predicted)
            )
            * 100
        ),
    }


persistence_validation_predictions = validation_data[
    "Food_Inflation_Lag1_Pct"
].to_numpy()

persistence_metrics = calculate_regression_metrics(
    validation_target,
    persistence_validation_predictions,
)

validation_model_results = pd.DataFrame(
    [
        {
            "Model": "Persistence",
            "Representation": "Lag-1 benchmark",
            **persistence_metrics,
        }
    ]
)

display(validation_model_results)

,Model,Representation,MAE,RMSE,R2,Directional_Accuracy_Pct
0,Persistence,Lag-1 benchmark,1.352852,2.340242,-0.519088,56.340580


### Persistence Benchmark Interpretation

The persistence benchmark produces a validation RMSE of 2.340 and a negative R² value.

The negative R² indicates that repeating the previous month's inflation rate performs worse than using the validation-period mean as a constant prediction. Monthly subclass inflation therefore does not follow a sufficiently stable month-to-month persistence pattern for the lag-one value to provide a strong standalone forecast.

Directional accuracy is approximately 56%, showing only a modest ability to predict whether monthly food inflation will be positive or negative.

The persistence results establish a deliberately simple reference point. Subsequent models should reduce both MAE and RMSE and preferably produce a positive validation R².

In [75]:
# tune ridge regression

ridge_alpha_values = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]


def tune_ridge_models(
    training_features,
    validation_features,
    training_target,
    validation_target,
    preprocessors,
    alpha_values,
):
    records = []
    selected_models = {}

    for representation in training_features:
        best_rmse = np.inf
        best_model = None

        for alpha in alpha_values:
            model_pipeline = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        clone(preprocessors[representation]),
                    ),
                    (
                        "model",
                        Ridge(alpha=alpha),
                    ),
                ]
            )

            model_pipeline.fit(
                training_features[representation],
                training_target,
            )

            predictions = model_pipeline.predict(
                validation_features[representation]
            )

            metrics = calculate_regression_metrics(
                validation_target,
                predictions,
            )

            records.append(
                {
                    "Model": "Ridge",
                    "Representation": representation,
                    "Alpha": alpha,
                    **metrics,
                }
            )

            if metrics["RMSE"] < best_rmse:
                best_rmse = metrics["RMSE"]
                best_model = model_pipeline

        selected_models[representation] = best_model

    return pd.DataFrame(records), selected_models


ridge_tuning_results, selected_ridge_models = (
    tune_ridge_models(
        training_features=training_features,
        validation_features=validation_features,
        training_target=training_target,
        validation_target=validation_target,
        preprocessors=ridge_preprocessors,
        alpha_values=ridge_alpha_values,
    )
)

print(
    "Ridge candidates evaluated:",
    len(ridge_tuning_results),
)

Ridge candidates evaluated: 14


In [76]:
# select ridge models

selected_ridge_results = (
    ridge_tuning_results
    .sort_values(
        ["Representation", "RMSE", "MAE"]
    )
    .groupby(
        "Representation",
        as_index=False,
    )
    .first()
)

display(
    ridge_tuning_results.sort_values(
        ["Representation", "Alpha"]
    )
)

print("Selected Ridge specifications:")

display(selected_ridge_results)

validation_model_results = pd.concat(
    [
        validation_model_results,
        selected_ridge_results[
            [
                "Model",
                "Representation",
                "MAE",
                "RMSE",
                "R2",
                "Directional_Accuracy_Pct",
            ]
        ],
    ],
    ignore_index=True,
)

print("Validation comparison to date:")

display(
    validation_model_results.sort_values("RMSE")
)

,Model,Representation,Alpha,MAE,RMSE,R2,Directional_Accuracy_Pct
7,Ridge,Asymmetric,0.001000,1.125376,1.879770,0.019899,62.862319
8,Ridge,Asymmetric,0.010000,1.125375,1.879768,0.019900,62.862319
9,Ridge,Asymmetric,0.100000,1.125356,1.879750,0.019919,62.862319
10,Ridge,Asymmetric,1.000000,1.125172,1.879574,0.020103,62.862319
11,Ridge,Asymmetric,10.000000,1.123590,1.878085,0.021655,63.043478
12,Ridge,Asymmetric,100.000000,1.118287,1.873179,0.026759,63.586957
13,Ridge,Asymmetric,1000.000000,1.116682,1.874167,0.025732,62.681159
0,Ridge,Symmetric,0.001000,1.120662,1.875962,0.023865,62.862319
1,Ridge,Symmetric,0.010000,1.120660,1.875961,0.023867,62.862319
2,Ridge,Symmetric,0.100000,1.120640,1.875943,0.023886,62.862319


Selected Ridge specifications:


,Representation,Model,Alpha,MAE,RMSE,R2,Directional_Accuracy_Pct
0,Asymmetric,Ridge,100.000000,1.118287,1.873179,0.026759,63.586957
1,Symmetric,Ridge,100.000000,1.113336,1.869568,0.030509,63.768116


Validation comparison to date:


,Model,Representation,MAE,RMSE,R2,Directional_Accuracy_Pct
2,Ridge,Symmetric,1.113336,1.869568,0.030509,63.768116
1,Ridge,Asymmetric,1.118287,1.873179,0.026759,63.586957
0,Persistence,Lag-1 benchmark,1.352852,2.340242,-0.519088,56.340580


### Ridge Regression Interpretation

Both Ridge models outperform the persistence benchmark across MAE, RMSE, R² and directional accuracy.

The symmetric Ridge model provides the best validation performance, with an RMSE of 1.870 compared with 1.873 for the asymmetric model. The difference is small, but it does not indicate a predictive advantage from separating depreciation and appreciation within the linear specification.

Both representations select an alpha value of 100. This indicates that
coefficient shrinkage improves validation performance and helps control the effects of correlated lagged predictors.

The positive R² values represent an improvement over persistence, although they remain close to zero. This suggests that the linear models explain only a small proportion of the month-to-month variation in food inflation.

Tree-based models are evaluated next to determine whether nonlinear effects and interactions improve predictive performance.

In [77]:
# tune random forest models

random_forest_parameter_grid = {
    "max_depth": [
        None,
        8,
        16,
    ],
    "min_samples_leaf": [
        1,
        3,
        6,
    ],
    "max_features": [
        "sqrt",
        0.7,
    ],
}

random_forest_candidates = list(
    ParameterGrid(random_forest_parameter_grid)
)


def tune_random_forest_models(
    training_features,
    validation_features,
    training_target,
    validation_target,
    preprocessors,
    parameter_candidates,
):
    records = []
    selected_models = {}

    for representation in training_features:
        best_rmse = np.inf
        best_model = None

        for parameters in parameter_candidates:
            model_pipeline = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        clone(preprocessors[representation]),
                    ),
                    (
                        "model",
                        RandomForestRegressor(
                            n_estimators=300,
                            random_state=RANDOM_STATE,
                            n_jobs=-1,
                            **parameters,
                        ),
                    ),
                ]
            )

            model_pipeline.fit(
                training_features[representation],
                training_target,
            )

            predictions = model_pipeline.predict(
                validation_features[representation]
            )

            metrics = calculate_regression_metrics(
                validation_target,
                predictions,
            )

            records.append(
                {
                    "Model": "Random Forest",
                    "Representation": representation,
                    "N_Estimators": 300,
                    **parameters,
                    **metrics,
                }
            )

            if metrics["RMSE"] < best_rmse:
                best_rmse = metrics["RMSE"]
                best_model = model_pipeline

        selected_models[representation] = best_model

    return pd.DataFrame(records), selected_models


rf_tuning_results, selected_rf_models = (
    tune_random_forest_models(
        training_features=training_features,
        validation_features=validation_features,
        training_target=training_target,
        validation_target=validation_target,
        preprocessors=tree_preprocessors,
        parameter_candidates=random_forest_candidates,
    )
)

print(
    "Random Forest candidates evaluated:",
    len(rf_tuning_results),
)

Random Forest candidates evaluated: 36


In [78]:
# select random forest models

ranked_rf_results = (
    rf_tuning_results
    .sort_values(
        ["Representation", "RMSE", "MAE"]
    )
    .reset_index(drop=True)
)

selected_rf_results = (
    ranked_rf_results
    .drop_duplicates(
        subset="Representation",
        keep="first",
    )
    .sort_values("Representation")
    .reset_index(drop=True)
)

print("Top Random Forest candidates:")

display(
    ranked_rf_results
    .groupby(
        "Representation",
        group_keys=False,
    )
    .head(5)
)

print("Selected Random Forest specifications:")

selected_rf_display = selected_rf_results.copy()

selected_rf_display["max_depth"] = (
    selected_rf_display["max_depth"]
    .astype(object)
    .where(
        selected_rf_display["max_depth"].notna(),
        "None",
    )
)

display(selected_rf_display)

selected_model_checks = []

for representation, model_pipeline in selected_rf_models.items():
    fitted_model = model_pipeline.named_steps["model"]

    selected_model_checks.append(
        {
            "Representation": representation,
            "Fitted_Max_Depth": (
                fitted_model.max_depth
                if fitted_model.max_depth is not None
                else "None"
            ),
            "Fitted_Max_Features": (
                fitted_model.max_features
            ),
            "Fitted_Min_Samples_Leaf": (
                fitted_model.min_samples_leaf
            ),
        }
    )

print("Fitted pipeline settings:")

display(pd.DataFrame(selected_model_checks))

# Make the cell safe to rerun
validation_model_results = (
    validation_model_results.loc[
        validation_model_results["Model"]
        != "Random Forest"
    ]
    .copy()
)

validation_model_results = pd.concat(
    [
        validation_model_results,
        selected_rf_results[
            [
                "Model",
                "Representation",
                "MAE",
                "RMSE",
                "R2",
                "Directional_Accuracy_Pct",
            ]
        ],
    ],
    ignore_index=True,
)

print("Validation comparison to date:")

display(
    validation_model_results.sort_values("RMSE")
)

Top Random Forest candidates:


,Model,Representation,N_Estimators,max_depth,max_features,min_samples_leaf,MAE,RMSE,R2,Directional_Accuracy_Pct
0,Random Forest,Asymmetric,300,16.000000,sqrt,1,1.077349,1.796596,0.104713,63.949275
1,Random Forest,Asymmetric,300,NaN,sqrt,1,1.077641,1.803326,0.097993,64.130435
2,Random Forest,Asymmetric,300,NaN,sqrt,3,1.079497,1.810950,0.090349,63.586957
3,Random Forest,Asymmetric,300,16.000000,sqrt,3,1.083886,1.820647,0.080582,63.586957
4,Random Forest,Asymmetric,300,16.000000,0.700000,6,1.084117,1.837942,0.063031,63.405797
18,Random Forest,Symmetric,300,NaN,sqrt,1,1.059113,1.759446,0.141356,64.855072
19,Random Forest,Symmetric,300,16.000000,sqrt,1,1.061006,1.769358,0.131653,64.130435
20,Random Forest,Symmetric,300,NaN,sqrt,3,1.063629,1.797580,0.103732,64.492754
21,Random Forest,Symmetric,300,16.000000,sqrt,3,1.068258,1.798703,0.102611,63.949275
22,Random Forest,Symmetric,300,8.000000,sqrt,1,1.089829,1.816184,0.085084,63.586957


Selected Random Forest specifications:


,Model,Representation,N_Estimators,max_depth,max_features,min_samples_leaf,MAE,RMSE,R2,Directional_Accuracy_Pct
0,Random Forest,Asymmetric,300,16.000000,sqrt,1,1.077349,1.796596,0.104713,63.949275
1,Random Forest,Symmetric,300,None,sqrt,1,1.059113,1.759446,0.141356,64.855072


Fitted pipeline settings:


,Representation,Fitted_Max_Depth,Fitted_Max_Features,Fitted_Min_Samples_Leaf
0,Symmetric,None,sqrt,1
1,Asymmetric,16,sqrt,1


Validation comparison to date:


,Model,Representation,MAE,RMSE,R2,Directional_Accuracy_Pct
4,Random Forest,Symmetric,1.059113,1.759446,0.141356,64.855072
3,Random Forest,Asymmetric,1.077349,1.796596,0.104713,63.949275
2,Ridge,Symmetric,1.113336,1.869568,0.030509,63.768116
1,Ridge,Asymmetric,1.118287,1.873179,0.026759,63.586957
0,Persistence,Lag-1 benchmark,1.352852,2.340242,-0.519088,56.340580


### Random Forest Interpretation

Both Random Forest models outperform the persistence and Ridge benchmarks, indicating that nonlinear relationships and interactions provide additional predictive value.

The symmetric Random Forest produces the strongest validation performance, with an RMSE of 1.759 and an R² of 0.141. Its selected specification uses unrestricted tree depth, square-root feature sampling and a minimum leaf size of one observation.

The asymmetric Random Forest achieves an RMSE of 1.797 and an R² of 0.105. Although it improves upon both Ridge models, it performs worse than the symmetric Random Forest.

Separating depreciation and appreciation therefore does not improve Random Forest validation performance. This result does not rule out category-specific asymmetry, but it suggests that the additional asymmetric predictors do not improve pooled validation forecasts for this model family.

XGBoost is evaluated next to determine whether sequential boosting and stronger regularisation improve the modelling of monthly food inflation.

In [79]:
# tune XGBoost models

xgboost_parameter_grid = {
    "learning_rate": [
        0.03,
        0.08,
    ],
    "max_depth": [
        2,
        4,
        6,
    ],
    "subsample": [
        0.8,
        1.0,
    ],
    "colsample_bytree": [
        0.8,
        1.0,
    ],
}

xgboost_candidates = list(
    ParameterGrid(xgboost_parameter_grid)
)


def tune_xgboost_models(
    training_features,
    validation_features,
    training_target,
    validation_target,
    preprocessors,
    parameter_candidates,
):
    records = []
    selected_models = {}

    for representation in training_features:
        best_rmse = np.inf
        best_model = None

        for parameters in parameter_candidates:
            model_pipeline = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        clone(preprocessors[representation]),
                    ),
                    (
                        "model",
                        XGBRegressor(
                            objective="reg:squarederror",
                            eval_metric="rmse",
                            n_estimators=300,
                            min_child_weight=3,
                            reg_lambda=1.0,
                            tree_method="hist",
                            random_state=RANDOM_STATE,
                            n_jobs=-1,
                            verbosity=0,
                            **parameters,
                        ),
                    ),
                ]
            )

            model_pipeline.fit(
                training_features[representation],
                training_target,
            )

            predictions = model_pipeline.predict(
                validation_features[representation]
            )

            metrics = calculate_regression_metrics(
                validation_target,
                predictions,
            )

            records.append(
                {
                    "Model": "XGBoost",
                    "Representation": representation,
                    "N_Estimators": 300,
                    "Min_Child_Weight": 3,
                    "Reg_Lambda": 1.0,
                    **parameters,
                    **metrics,
                }
            )

            if metrics["RMSE"] < best_rmse:
                best_rmse = metrics["RMSE"]
                best_model = model_pipeline

        selected_models[representation] = best_model

    return pd.DataFrame(records), selected_models


xgboost_tuning_results, selected_xgboost_models = (
    tune_xgboost_models(
        training_features=training_features,
        validation_features=validation_features,
        training_target=training_target,
        validation_target=validation_target,
        preprocessors=tree_preprocessors,
        parameter_candidates=xgboost_candidates,
    )
)

print(
    "XGBoost candidates evaluated:",
    len(xgboost_tuning_results),
)

XGBoost candidates evaluated: 48


In [80]:
# select XGBoost models

ranked_xgboost_results = (
    xgboost_tuning_results
    .sort_values(
        ["Representation", "RMSE", "MAE"]
    )
    .reset_index(drop=True)
)

selected_xgboost_results = (
    ranked_xgboost_results
    .drop_duplicates(
        subset="Representation",
        keep="first",
    )
    .sort_values("Representation")
    .reset_index(drop=True)
)

print("Top XGBoost candidates:")

display(
    ranked_xgboost_results
    .groupby(
        "Representation",
        group_keys=False,
    )
    .head(5)
)

print("Selected XGBoost specifications:")

display(selected_xgboost_results)

# Make the cell safe to rerun
validation_model_results = (
    validation_model_results.loc[
        validation_model_results["Model"]
        != "XGBoost"
    ]
    .copy()
)

validation_model_results = pd.concat(
    [
        validation_model_results,
        selected_xgboost_results[
            [
                "Model",
                "Representation",
                "MAE",
                "RMSE",
                "R2",
                "Directional_Accuracy_Pct",
            ]
        ],
    ],
    ignore_index=True,
)

print("Complete validation comparison:")

display(
    validation_model_results.sort_values("RMSE")
)

Top XGBoost candidates:


,Model,Representation,N_Estimators,Min_Child_Weight,Reg_Lambda,colsample_bytree,learning_rate,max_depth,subsample,MAE,RMSE,R2,Directional_Accuracy_Pct
0,XGBoost,Asymmetric,300,3,1.000000,1.000000,0.080000,4,0.800000,1.073955,1.767278,0.133694,63.405797
1,XGBoost,Asymmetric,300,3,1.000000,1.000000,0.030000,6,0.800000,1.063523,1.771198,0.129846,64.673913
2,XGBoost,Asymmetric,300,3,1.000000,0.800000,0.080000,4,0.800000,1.068524,1.772210,0.128852,63.949275
3,XGBoost,Asymmetric,300,3,1.000000,0.800000,0.030000,6,0.800000,1.066156,1.772821,0.128252,63.768116
4,XGBoost,Asymmetric,300,3,1.000000,0.800000,0.080000,6,1.000000,1.099587,1.783150,0.118064,63.043478
24,XGBoost,Symmetric,300,3,1.000000,1.000000,0.080000,4,0.800000,1.062493,1.757873,0.142891,64.311594
25,XGBoost,Symmetric,300,3,1.000000,0.800000,0.080000,4,0.800000,1.063008,1.777469,0.123675,62.862319
26,XGBoost,Symmetric,300,3,1.000000,0.800000,0.030000,6,0.800000,1.059566,1.780410,0.120771,63.043478
27,XGBoost,Symmetric,300,3,1.000000,0.800000,0.080000,6,0.800000,1.090440,1.784105,0.117118,62.137681
28,XGBoost,Symmetric,300,3,1.000000,1.000000,0.030000,6,0.800000,1.065803,1.789722,0.111550,64.311594


Selected XGBoost specifications:


,Model,Representation,N_Estimators,Min_Child_Weight,Reg_Lambda,colsample_bytree,learning_rate,max_depth,subsample,MAE,RMSE,R2,Directional_Accuracy_Pct
0,XGBoost,Asymmetric,300,3,1.000000,1.000000,0.080000,4,0.800000,1.073955,1.767278,0.133694,63.405797
1,XGBoost,Symmetric,300,3,1.000000,1.000000,0.080000,4,0.800000,1.062493,1.757873,0.142891,64.311594


Complete validation comparison:


,Model,Representation,MAE,RMSE,R2,Directional_Accuracy_Pct
6,XGBoost,Symmetric,1.062493,1.757873,0.142891,64.311594
4,Random Forest,Symmetric,1.059113,1.759446,0.141356,64.855072
5,XGBoost,Asymmetric,1.073955,1.767278,0.133694,63.405797
3,Random Forest,Asymmetric,1.077349,1.796596,0.104713,63.949275
2,Ridge,Symmetric,1.113336,1.869568,0.030509,63.768116
1,Ridge,Asymmetric,1.118287,1.873179,0.026759,63.586957
0,Persistence,Lag-1 benchmark,1.352852,2.340242,-0.519088,56.340580


### Initial XGBoost Interpretation

Both XGBoost models outperform Ridge regression and the persistence benchmark.

The symmetric XGBoost model produces the lowest validation RMSE of 1.758, although its advantage over the symmetric Random Forest is very small. Random Forest retains slightly better MAE and directional accuracy.

The asymmetric XGBoost model outperforms the asymmetric Random Forest, but it does not outperform either of the leading symmetric tree models. As with Ridge and Random Forest, separating depreciation and appreciation does not improve validation RMSE.

Both XGBoost representations select a learning rate of 0.08, maximum depth of four, row subsampling of 0.8 and full feature availability. Because several of these values occur at the boundaries of the initial search, a compact local refinement is performed before final model selection.

In [81]:
# 9.4.1 define rfinement candidates

base_xgboost_parameters = {
    "n_estimators": 300,
    "learning_rate": 0.08,
    "max_depth": 4,
    "subsample": 0.8,
    "colsample_bytree": 1.0,
    "min_child_weight": 3,
    "reg_lambda": 1.0,
}

xgboost_parameter_variations = [
    {},
    {
        "n_estimators": 450,
        "learning_rate": 0.05,
    },
    {
        "n_estimators": 200,
        "learning_rate": 0.12,
    },
    {"max_depth": 3},
    {"max_depth": 5},
    {"subsample": 0.7},
    {"subsample": 0.9},
    {"colsample_bytree": 0.9},
    {"min_child_weight": 1},
    {"min_child_weight": 5},
    {"reg_lambda": 0.5},
    {"reg_lambda": 5.0},
]

xgboost_refinement_candidates = [
    {
        **base_xgboost_parameters,
        **variation,
    }
    for variation in xgboost_parameter_variations
]

print(
    "Refinement candidates per representation:",
    len(xgboost_refinement_candidates),
)

Refinement candidates per representation: 12


In [82]:
# run XGBoost refinement

def refine_xgboost_models(
    training_features,
    validation_features,
    training_target,
    validation_target,
    preprocessors,
    parameter_candidates,
):
    records = []

    for representation in training_features:
        for parameters in parameter_candidates:
            model_pipeline = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        clone(preprocessors[representation]),
                    ),
                    (
                        "model",
                        XGBRegressor(
                            objective="reg:squarederror",
                            eval_metric="rmse",
                            tree_method="hist",
                            random_state=RANDOM_STATE,
                            n_jobs=-1,
                            verbosity=0,
                            **parameters,
                        ),
                    ),
                ]
            )

            model_pipeline.fit(
                training_features[representation],
                training_target,
            )

            predictions = model_pipeline.predict(
                validation_features[representation]
            )

            metrics = calculate_regression_metrics(
                validation_target,
                predictions,
            )

            records.append(
                {
                    "Model": "XGBoost",
                    "Representation": representation,
                    "N_Estimators": parameters[
                        "n_estimators"
                    ],
                    "Min_Child_Weight": parameters[
                        "min_child_weight"
                    ],
                    "Reg_Lambda": parameters["reg_lambda"],
                    "colsample_bytree": parameters[
                        "colsample_bytree"
                    ],
                    "learning_rate": parameters[
                        "learning_rate"
                    ],
                    "max_depth": parameters["max_depth"],
                    "subsample": parameters["subsample"],
                    **metrics,
                }
            )

    return pd.DataFrame(records)


xgboost_refinement_results = refine_xgboost_models(
    training_features=training_features,
    validation_features=validation_features,
    training_target=training_target,
    validation_target=validation_target,
    preprocessors=tree_preprocessors,
    parameter_candidates=xgboost_refinement_candidates,
)

print(
    "XGBoost refinement models evaluated:",
    len(xgboost_refinement_results),
)

XGBoost refinement models evaluated: 24


In [83]:
# finalise XGBoost selection

xgboost_parameter_columns = [
    "Representation",
    "N_Estimators",
    "Min_Child_Weight",
    "Reg_Lambda",
    "colsample_bytree",
    "learning_rate",
    "max_depth",
    "subsample",
]

combined_xgboost_tuning_results = (
    pd.concat(
        [
            xgboost_tuning_results,
            xgboost_refinement_results,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=xgboost_parameter_columns,
        keep="first",
    )
    .sort_values(
        ["Representation", "RMSE", "MAE"]
    )
    .reset_index(drop=True)
)

selected_xgboost_results = (
    combined_xgboost_tuning_results
    .drop_duplicates(
        subset="Representation",
        keep="first",
    )
    .sort_values("Representation")
    .reset_index(drop=True)
)

selected_xgboost_models = {}

for row in selected_xgboost_results.itertuples(
    index=False
):
    representation = row.Representation

    selected_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(tree_preprocessors[representation]),
            ),
            (
                "model",
                XGBRegressor(
                    objective="reg:squarederror",
                    eval_metric="rmse",
                    tree_method="hist",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbosity=0,
                    n_estimators=int(row.N_Estimators),
                    min_child_weight=(
                        row.Min_Child_Weight
                    ),
                    reg_lambda=row.Reg_Lambda,
                    colsample_bytree=(
                        row.colsample_bytree
                    ),
                    learning_rate=row.learning_rate,
                    max_depth=int(row.max_depth),
                    subsample=row.subsample,
                ),
            ),
        ]
    )

    selected_pipeline.fit(
        training_features[representation],
        training_target,
    )

    selected_xgboost_models[
        representation
    ] = selected_pipeline

print("Top refined XGBoost candidates:")

display(
    combined_xgboost_tuning_results
    .groupby(
        "Representation",
        group_keys=False,
    )
    .head(5)
)

print("Final selected XGBoost specifications:")

display(selected_xgboost_results)

validation_model_results = (
    validation_model_results.loc[
        validation_model_results["Model"]
        != "XGBoost"
    ]
    .copy()
)

validation_model_results = pd.concat(
    [
        validation_model_results,
        selected_xgboost_results[
            [
                "Model",
                "Representation",
                "MAE",
                "RMSE",
                "R2",
                "Directional_Accuracy_Pct",
            ]
        ],
    ],
    ignore_index=True,
)

print("Updated validation comparison:")

display(
    validation_model_results.sort_values("RMSE")
)

Top refined XGBoost candidates:


,Model,Representation,N_Estimators,Min_Child_Weight,Reg_Lambda,colsample_bytree,learning_rate,max_depth,subsample,MAE,RMSE,R2,Directional_Accuracy_Pct
0,XGBoost,Asymmetric,300,3,0.500000,1.000000,0.080000,4,0.800000,1.058929,1.728267,0.171518,63.405797
1,XGBoost,Asymmetric,300,3,1.000000,1.000000,0.080000,4,0.700000,1.063132,1.748867,0.151650,63.768116
2,XGBoost,Asymmetric,300,3,5.000000,1.000000,0.080000,4,0.800000,1.059544,1.760538,0.140290,63.768116
3,XGBoost,Asymmetric,300,5,1.000000,1.000000,0.080000,4,0.800000,1.064631,1.762109,0.138755,63.405797
4,XGBoost,Asymmetric,300,3,1.000000,1.000000,0.080000,5,0.800000,1.085448,1.763136,0.137750,62.862319
35,XGBoost,Symmetric,300,3,0.500000,1.000000,0.080000,4,0.800000,1.046677,1.722070,0.177449,63.949275
36,XGBoost,Symmetric,300,3,1.000000,1.000000,0.080000,4,0.700000,1.051723,1.736607,0.163502,63.768116
37,XGBoost,Symmetric,450,3,1.000000,1.000000,0.050000,4,0.800000,1.057593,1.749175,0.151351,63.949275
38,XGBoost,Symmetric,300,3,1.000000,1.000000,0.080000,5,0.800000,1.070391,1.751050,0.149530,61.956522
39,XGBoost,Symmetric,300,5,1.000000,1.000000,0.080000,4,0.800000,1.066683,1.752102,0.148508,63.043478


Final selected XGBoost specifications:


,Model,Representation,N_Estimators,Min_Child_Weight,Reg_Lambda,colsample_bytree,learning_rate,max_depth,subsample,MAE,RMSE,R2,Directional_Accuracy_Pct
0,XGBoost,Asymmetric,300,3,0.500000,1.000000,0.080000,4,0.800000,1.058929,1.728267,0.171518,63.405797
1,XGBoost,Symmetric,300,3,0.500000,1.000000,0.080000,4,0.800000,1.046677,1.722070,0.177449,63.949275


Updated validation comparison:


,Model,Representation,MAE,RMSE,R2,Directional_Accuracy_Pct
6,XGBoost,Symmetric,1.046677,1.722070,0.177449,63.949275
5,XGBoost,Asymmetric,1.058929,1.728267,0.171518,63.405797
4,Random Forest,Symmetric,1.059113,1.759446,0.141356,64.855072
3,Random Forest,Asymmetric,1.077349,1.796596,0.104713,63.949275
2,Ridge,Symmetric,1.113336,1.869568,0.030509,63.768116
1,Ridge,Asymmetric,1.118287,1.873179,0.026759,63.586957
0,Persistence,Lag-1 benchmark,1.352852,2.340242,-0.519088,56.340580


### Final XGBoost Interpretation

The local refinement improves both XGBoost representations. Reducing the L2 regularisation parameter from 1.0 to 0.5 produces the best validation result for both feature sets.

The symmetric XGBoost model achieves the lowest validation RMSE of 1.722 and the highest R² of 0.177. The asymmetric model follows closely with an RMSE of 1.728 and an R² of 0.172.

The symmetric model therefore remains preferable under the primary
validation-selection metric. However, the difference between the two XGBoost representations is small and must be reassessed using the untouched test period.

Random Forest retains the highest directional accuracy, while XGBoost provides the strongest MAE, RMSE and R² performance. Model selection is based on RMSE, as specified before tuning.

In [84]:
# compare feature representations

representation_comparison = (
    validation_model_results.loc[
        validation_model_results["Model"]
        != "Persistence"
    ]
    .pivot(
        index="Model",
        columns="Representation",
        values="RMSE",
    )
    .reset_index()
)

representation_comparison[
    "Asymmetric_Improvement_Pct"
] = (
    (
        representation_comparison["Symmetric"]
        - representation_comparison["Asymmetric"]
    )
    / representation_comparison["Symmetric"]
    * 100
)

representation_comparison[
    "Preferred_Representation"
] = np.where(
    representation_comparison[
        "Asymmetric_Improvement_Pct"
    ] > 0,
    "Asymmetric",
    "Symmetric",
)

display(
    representation_comparison.sort_values(
        "Asymmetric_Improvement_Pct",
        ascending=False,
    )
)

selected_validation_model = (
    validation_model_results
    .sort_values(["RMSE", "MAE"])
    .iloc[0]
)

print("Leading validation model:")
print(
    f"{selected_validation_model['Model']} — "
    f"{selected_validation_model['Representation']}"
)
print(
    "Validation RMSE:",
    round(selected_validation_model["RMSE"], 6),
)

Representation,Model,Asymmetric,Symmetric,Asymmetric_Improvement_Pct,Preferred_Representation
1,Ridge,1.873179,1.869568,-0.193184,Symmetric
2,XGBoost,1.728267,1.722070,-0.359888,Symmetric
0,Random Forest,1.796596,1.759446,-2.111485,Symmetric


Leading validation model:
XGBoost — Symmetric
Validation RMSE: 1.72207


In [85]:
# create final development matrices

development_data = (
    pd.concat(
        [
            train_data,
            validation_data,
        ],
        ignore_index=True,
    )
    .sort_values(["Date", "SubclassDescription"])
    .reset_index(drop=True)
)

development_features = {
    representation: development_data[
        features
    ].copy()
    for representation, features
    in model_feature_sets.items()
}

test_features = {
    representation: test_data[features].copy()
    for representation, features
    in model_feature_sets.items()
}

development_target = development_data[
    target_column
].copy()

final_sample_summary = pd.DataFrame(
    {
        "Sample": [
            "Development",
            "Locked test",
        ],
        "Start_Date": [
            development_data["Date"].min(),
            test_data["Date"].min(),
        ],
        "End_Date": [
            development_data["Date"].max(),
            test_data["Date"].max(),
        ],
        "Observations": [
            len(development_data),
            len(test_data),
        ],
        "Food_Subclasses": [
            development_data[
                "SubclassDescription"
            ].nunique(),
            test_data[
                "SubclassDescription"
            ].nunique(),
        ],
        "Unique_Months": [
            development_data["Date"].nunique(),
            test_data["Date"].nunique(),
        ],
    }
)

display(final_sample_summary)

,Sample,Start_Date,End_Date,Observations,Food_Subclasses,Unique_Months
0,Development,2017-10-01,2024-12-01,4002,46,87
1,Locked test,2025-01-01,2025-12-01,552,46,12


In [86]:
# refit models and generate predictions

selected_model_templates = {
    ("Ridge", "Symmetric"): (
        selected_ridge_models["Symmetric"]
    ),
    ("Ridge", "Asymmetric"): (
        selected_ridge_models["Asymmetric"]
    ),
    ("Random Forest", "Symmetric"): (
        selected_rf_models["Symmetric"]
    ),
    ("Random Forest", "Asymmetric"): (
        selected_rf_models["Asymmetric"]
    ),
    ("XGBoost", "Symmetric"): (
        selected_xgboost_models["Symmetric"]
    ),
    ("XGBoost", "Asymmetric"): (
        selected_xgboost_models["Asymmetric"]
    ),
}

final_fitted_models = {}
prediction_tables = []

test_metadata = test_data[
    [
        "Date",
        "ClassDescription",
        "SubclassDescription",
    ]
].reset_index(drop=True)

for (
    model_name,
    representation,
), model_template in selected_model_templates.items():
    final_model = clone(model_template)

    final_model.fit(
        development_features[representation],
        development_target,
    )

    predictions = final_model.predict(
        test_features[representation]
    )

    final_fitted_models[
        (model_name, representation)
    ] = final_model

    model_predictions = test_metadata.copy()
    model_predictions["Model"] = model_name
    model_predictions[
        "Representation"
    ] = representation
    model_predictions[
        "Predicted_Food_Inflation_Pct"
    ] = predictions

    prediction_tables.append(model_predictions)

persistence_predictions = test_metadata.copy()
persistence_predictions["Model"] = "Persistence"
persistence_predictions[
    "Representation"
] = "Lag-1 benchmark"
persistence_predictions[
    "Predicted_Food_Inflation_Pct"
] = test_data[
    "Food_Inflation_Lag1_Pct"
].to_numpy()

prediction_tables.append(persistence_predictions)

final_test_predictions = (
    pd.concat(
        prediction_tables,
        ignore_index=True,
    )
    .sort_values(
        [
            "Model",
            "Representation",
            "Date",
            "SubclassDescription",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Final fitted models:",
    len(final_fitted_models),
)
print(
    "Prediction rows:",
    len(final_test_predictions),
)
print(
    "Expected prediction rows:",
    len(test_data) * 7,
)
print(
    "Missing predictions:",
    final_test_predictions[
        "Predicted_Food_Inflation_Pct"
    ].isna().sum(),
)
print(
    "Duplicate prediction rows:",
    final_test_predictions.duplicated(
        [
            "Date",
            "SubclassDescription",
            "Model",
            "Representation",
        ]
    ).sum(),
)

display(
    final_test_predictions.groupby(
        ["Model", "Representation"]
    )
    .agg(
        Predictions=(
            "Predicted_Food_Inflation_Pct",
            "size",
        ),
        Start_Date=("Date", "min"),
        End_Date=("Date", "max"),
        Food_Subclasses=(
            "SubclassDescription",
            "nunique",
        ),
    )
)

Final fitted models: 6
Prediction rows: 3864
Expected prediction rows: 3864
Missing predictions: 0
Duplicate prediction rows: 0


Predictions Start_Date   End_Date  \
Model         Representation                                       
Persistence   Lag-1 benchmark          552 2025-01-01 2025-12-01   
Random Forest Asymmetric               552 2025-01-01 2025-12-01   
              Symmetric                552 2025-01-01 2025-12-01   
Ridge         Asymmetric               552 2025-01-01 2025-12-01   
              Symmetric                552 2025-01-01 2025-12-01   
XGBoost       Asymmetric               552 2025-01-01 2025-12-01   
              Symmetric                552 2025-01-01 2025-12-01   

                               Food_Subclasses  
Model         Representation                    
Persistence   Lag-1 benchmark               46  
Random Forest Asymmetric                    46  
              Symmetric                     46  
Ridge         Asymmetric                    46  
              Symmetric                     46  
XGBoost       Asymmetric                    46  
              Symmetric                     46

### Final Training Interpretation

The six selected machine learning pipelines were refitted using the combined training and validation sample. The final development period contains 87 months for each of the 46 food subclasses and ends in December 2024.

Each fitted pipeline produces 552 predictions for the 2025 test period.
Together with the persistence benchmark, the prediction dataset contains seven complete model variants and 3,864 prediction rows.

No test-period performance measure has been calculated. This preserves the test period for a single final evaluation in Notebook 9.

The prediction design represents rolling one-month-ahead forecasting. When predicting a particular month, lagged food inflation from preceding months is treated as information that would already be available. This differs from a fixed-origin twelve-month forecast, in which future lagged outcomes would not be observed.

In [87]:
# run leakage and output audit

excluded_metadata = {
    "Date",
    "Split",
    "ClassDescription",
    "Subclass_Weight",
}

all_selected_features = set(
    symmetric_model_features
    + asymmetric_model_features
)

exchange_rate_features = (
    symmetric_exchange_features
    + asymmetric_exchange_features
)

expected_model_keys = set(
    selected_model_templates.keys()
)

prediction_counts = (
    final_test_predictions
    .groupby(["Model", "Representation"])
    .size()
)

audit_records = [
    {
        "Check": "Current target excluded from features",
        "Passed": (
            target_column not in all_selected_features
        ),
    },
    {
        "Check": "Date excluded from features",
        "Passed": (
            "Date" not in all_selected_features
        ),
    },
    {
        "Check": "Split label excluded from features",
        "Passed": (
            "Split" not in all_selected_features
        ),
    },
    {
        "Check": "Descriptive metadata excluded",
        "Passed": not bool(
            excluded_metadata.intersection(
                all_selected_features
            )
        ),
    },
    {
        "Check": "Exchange-rate predictors are lagged",
        "Passed": all(
            "_Lag" in feature
            for feature in exchange_rate_features
        ),
    },
    {
        "Check": "Development ends before test period",
        "Passed": (
            development_data["Date"].max()
            < test_data["Date"].min()
        ),
    },
    {
        "Check": "Test targets excluded from predictions",
        "Passed": (
            target_column
            not in final_test_predictions.columns
        ),
    },
    {
        "Check": "All fitted model variants present",
        "Passed": (
            set(final_fitted_models.keys())
            == expected_model_keys
        ),
    },
    {
        "Check": "Every model has 552 predictions",
        "Passed": (
            prediction_counts.eq(
                len(test_data)
            ).all()
        ),
    },
    {
        "Check": "Prediction row count is correct",
        "Passed": (
            len(final_test_predictions)
            == len(test_data) * 7
        ),
    },
    {
        "Check": "All predictions are finite",
        "Passed": np.isfinite(
            final_test_predictions[
                "Predicted_Food_Inflation_Pct"
            ]
        ).all(),
    },
    {
        "Check": "No duplicate prediction records",
        "Passed": not final_test_predictions.duplicated(
            [
                "Date",
                "SubclassDescription",
                "Model",
                "Representation",
            ]
        ).any(),
    },
]

leakage_and_output_audit = pd.DataFrame(
    audit_records
)

display(leakage_and_output_audit)

print(
    "All leakage and output checks passed:",
    leakage_and_output_audit["Passed"].all(),
)

,Check,Passed
0,Current target excluded from features,True
1,Date excluded from features,True
2,Split label excluded from features,True
3,Descriptive metadata excluded,True
4,Exchange-rate predictors are lagged,True
5,Development ends before test period,True
6,Test targets excluded from predictions,True
7,All fitted model variants present,True
8,Every model has 552 predictions,True
9,Prediction row count is correct,True


All leakage and output checks passed: True


In [92]:
# create output directories

model_output_directory = Path("../models/machine_learning")
table_output_directory = Path("../reports/tables/machine_learning")

model_output_directory.mkdir(parents=True, exist_ok=True)
table_output_directory.mkdir(parents=True, exist_ok=True)


# save the six fitted pipelines
model_manifest_records = []

for (model_name, representation), fitted_pipeline in final_fitted_models.items():
    file_stem = (
        f"{model_name}_{representation}_pipeline"
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )
    model_file = model_output_directory / f"{file_stem}.joblib"

    joblib.dump(fitted_pipeline, model_file)

    estimator = fitted_pipeline.named_steps["model"]

    validation_row = validation_model_results.loc[
        (validation_model_results["Model"] == model_name)
        & (
            validation_model_results["Representation"]
            == representation
        )
    ].iloc[0]

    model_manifest_records.append(
        {
            "Model": model_name,
            "Representation": representation,
            "File": model_file.name,
            "Development_End_Date": development_data["Date"].max(),
            "Test_Start_Date": test_data["Date"].min(),
            "Validation_MAE": validation_row["MAE"],
            "Validation_RMSE": validation_row["RMSE"],
            "Validation_R2": validation_row["R2"],
            "Validation_Directional_Accuracy_Pct": validation_row[
                "Directional_Accuracy_Pct"
            ],
            "Estimator_Parameters": json.dumps(
                estimator.get_params(),
                default=str,
                sort_keys=True,
            ),
            "Saved": model_file.exists(),
        }
    )

model_manifest = pd.DataFrame(model_manifest_records)


# save tuning, selection, prediction, and audit tables
tables_to_export = {
    "ridge_tuning_results.csv": ridge_tuning_results,
    "selected_ridge_models.csv": selected_ridge_results,
    "random_forest_tuning_results.csv": rf_tuning_results,
    "selected_random_forest_models.csv": selected_rf_results,
    "xgboost_tuning_results.csv": combined_xgboost_tuning_results,
    "selected_xgboost_models.csv": selected_xgboost_results,
    "validation_model_comparison.csv": validation_model_results,
    "feature_representation_comparison.csv": representation_comparison,
    "leakage_and_output_audit.csv": leakage_and_output_audit,
    "ml_test_predictions.csv": final_test_predictions,
    "model_manifest.csv": model_manifest,
}

export_records = []

for file_name, table in tables_to_export.items():
    output_file = table_output_directory / file_name
    table.to_csv(output_file, index=False)

    export_records.append(
        {
            "File": file_name,
            "Rows": len(table),
            "Columns": len(table.columns),
            "Saved": output_file.exists(),
        }
    )

export_summary = pd.DataFrame(export_records)

print(f"Fitted pipelines saved: {model_manifest['Saved'].sum()}")
display(model_manifest.drop(columns="Estimator_Parameters"))
display(export_summary)

print(
    "All model files saved:",
    bool(model_manifest["Saved"].all()),
)
print(
    "All result tables saved:",
    bool(export_summary["Saved"].all()),
)

Fitted pipelines saved: 6


,Model,Representation,File,Development_End_Date,Test_Start_Date,Validation_MAE,Validation_RMSE,Validation_R2,Validation_Directional_Accuracy_Pct,Saved
0,Ridge,Symmetric,ridge_symmetric_pipeline.joblib,2024-12-01,2025-01-01,1.113336,1.869568,0.030509,63.768116,True
1,Ridge,Asymmetric,ridge_asymmetric_pipeline.joblib,2024-12-01,2025-01-01,1.118287,1.873179,0.026759,63.586957,True
2,Random Forest,Symmetric,random_forest_symmetric_pipeline.joblib,2024-12-01,2025-01-01,1.059113,1.759446,0.141356,64.855072,True
3,Random Forest,Asymmetric,random_forest_asymmetric_pipeline.joblib,2024-12-01,2025-01-01,1.077349,1.796596,0.104713,63.949275,True
4,XGBoost,Symmetric,xgboost_symmetric_pipeline.joblib,2024-12-01,2025-01-01,1.046677,1.722070,0.177449,63.949275,True
5,XGBoost,Asymmetric,xgboost_asymmetric_pipeline.joblib,2024-12-01,2025-01-01,1.058929,1.728267,0.171518,63.405797,True


,File,Rows,Columns,Saved
0,ridge_tuning_results.csv,14,7,True
1,selected_ridge_models.csv,2,7,True
2,random_forest_tuning_results.csv,36,10,True
3,selected_random_forest_models.csv,2,10,True
4,xgboost_tuning_results.csv,70,13,True
5,selected_xgboost_models.csv,2,13,True
6,validation_model_comparison.csv,7,6,True
7,feature_representation_comparison.csv,3,5,True
8,leakage_and_output_audit.csv,12,2,True
9,ml_test_predictions.csv,3864,6,True


All model files saved: True
All result tables saved: True


In [93]:
# validate exported tables
table_validation_records = []

for file_name, expected_table in tables_to_export.items():
    saved_file = table_output_directory / file_name
    saved_table = pd.read_csv(saved_file)

    table_validation_records.append(
        {
            "File": file_name,
            "Expected_Rows": len(expected_table),
            "Saved_Rows": len(saved_table),
            "Rows_Match": len(saved_table) == len(expected_table),
            "Expected_Columns": len(expected_table.columns),
            "Saved_Columns": len(saved_table.columns),
            "Columns_Match": (
                list(saved_table.columns)
                == list(expected_table.columns)
            ),
        }
    )

table_validation = pd.DataFrame(table_validation_records)

display(table_validation)

print(
    "All exported tables validated:",
    bool(
        table_validation[
            ["Rows_Match", "Columns_Match"]
        ].all().all()
    ),
)

saved_predictions = pd.read_csv(
    table_output_directory / "ml_test_predictions.csv"
)

print(
    "Current test target excluded:",
    "Food_Inflation_Pct" not in saved_predictions.columns,
)
print(
    "Prediction rows validated:",
    len(saved_predictions) == 3864,
)

,File,Expected_Rows,Saved_Rows,Rows_Match,Expected_Columns,Saved_Columns,Columns_Match
0,ridge_tuning_results.csv,14,14,True,7,7,True
1,selected_ridge_models.csv,2,2,True,7,7,True
2,random_forest_tuning_results.csv,36,36,True,10,10,True
3,selected_random_forest_models.csv,2,2,True,10,10,True
4,xgboost_tuning_results.csv,70,70,True,13,13,True
5,selected_xgboost_models.csv,2,2,True,13,13,True
6,validation_model_comparison.csv,7,7,True,6,6,True
7,feature_representation_comparison.csv,3,3,True,5,5,True
8,leakage_and_output_audit.csv,12,12,True,2,2,True
9,ml_test_predictions.csv,3864,3864,True,6,6,True


All exported tables validated: True
Current test target excluded: True
Prediction rows validated: True


In [94]:
# confirm that saved pipelines reproduce the predictions

identifier_columns = [
    "Date",
    "ClassDescription",
    "SubclassDescription",
]

model_reload_records = []

for _, manifest_row in model_manifest.iterrows():
    model_name = manifest_row["Model"]
    representation = manifest_row["Representation"]
    model_file = model_output_directory / manifest_row["File"]

    reloaded_pipeline = joblib.load(model_file)
    input_columns = list(reloaded_pipeline.feature_names_in_)

    reloaded_predictions = reloaded_pipeline.predict(
        test_data[input_columns]
    )

    expected_predictions = final_test_predictions.loc[
        (final_test_predictions["Model"] == model_name)
        & (
            final_test_predictions["Representation"]
            == representation
        ),
        identifier_columns + ["Predicted_Food_Inflation_Pct"],
    ].copy()

    prediction_check = test_data[identifier_columns].copy()
    prediction_check["Reloaded_Prediction"] = reloaded_predictions

    prediction_check = prediction_check.merge(
        expected_predictions,
        on=identifier_columns,
        how="inner",
        validate="one_to_one",
    )

    maximum_difference = np.max(
        np.abs(
            prediction_check["Reloaded_Prediction"]
            - prediction_check["Predicted_Food_Inflation_Pct"]
        )
    )

    model_reload_records.append(
        {
            "Model": model_name,
            "Representation": representation,
            "Expected_Predictions": 552,
            "Matched_Predictions": len(prediction_check),
            "Maximum_Prediction_Difference": maximum_difference,
            "Predictions_Reproduced": (
                len(prediction_check) == 552
                and np.isclose(maximum_difference, 0.0)
            ),
        }
    )

model_reload_validation = pd.DataFrame(model_reload_records)

display(model_reload_validation)

print(
    "All saved pipelines validated:",
    bool(model_reload_validation["Predictions_Reproduced"].all()),
)

,Model,Representation,Expected_Predictions,Matched_Predictions,Maximum_Prediction_Difference,Predictions_Reproduced
0,Ridge,Symmetric,552,552,0.000000,True
1,Ridge,Asymmetric,552,552,0.000000,True
2,Random Forest,Symmetric,552,552,0.000000,True
3,Random Forest,Asymmetric,552,552,0.000000,True
4,XGBoost,Symmetric,552,552,0.000000,True
5,XGBoost,Asymmetric,552,552,0.000000,True


All saved pipelines validated: True
